[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/templates/blob/main/machine-learning/notebooks/00_eda.ipynb)

# 00 — Exploratory Data Analysis

**Purpose.** Understand the data well enough to specify notebook 01: what the
columns mean, how they are distributed, where they are missing, how records are
grouped, and which values are impossible.

**Inputs.** `data/raw/` — read-only, never modified.

**Outputs.** *Insights, not artifacts.* Nothing here writes to `data/processed/`
or `models/`. The deliverable is section 11: a written specification of what
notebook 01 must do.

**Rules.**
- `data/raw/` is never modified.
- Findings here are hypotheses. Any threshold you *derive* from this data is a
  statistical threshold and belongs in a `02x` notebook, fitted on train only.
- A bound is only allowed in notebook 01 if you can name its external source
  (a standard, a specification, a physical limit). Record candidates in
  section 10 together with that source.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same
way they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the few packages Colab does not
already ship. Note that `data/` and `models/` are DVC-tracked and therefore *not*
part of the clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the template? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/templates.git"
BRANCH = "main"
SUBDIR = "machine-learning"  # project root inside the repository

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/<project-name>/data"
# CONFIG_OVERRIDES += [
#     f"raw_path={DATA_ROOT}/raw/dataset.parquet",
#     f"train_path={DATA_ROOT}/processed/train.parquet",
#     f"test_path={DATA_ROOT}/processed/test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)  # empty unless section 0 filled it in
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the raw data

Read-only. If the file is missing, run `task dvc:pull`.

In [ ]:
from src.data.load import load_raw

df = load_raw(cfg)
print(f"{len(df):,} records x {df.shape[1]} columns")
df.head()

## 3. Schema and dtypes

What is actually in the file, versus what `configs/data.yaml` declares. Any gap
found here is a change to the config, not a fix applied in code.

In [ ]:
df.info()

# TODO: compare the observed columns and dtypes against cfg.schema.columns and
# update configs/data.yaml until the declaration matches reality.
declared = set(cfg.schema.columns.keys())
observed = set(df.columns)
print("declared but absent:", sorted(declared - observed))
print("present but undeclared:", sorted(observed - declared))

## 4. Missingness

Where are the gaps, and are they random? A column missing 60% of the time, or
missing only for one group, is a modelling decision — not something to impute
without thinking.

Imputation itself happens in `02x`, fitted on train only.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

# TODO: for the worst columns, check whether missingness correlates with the
# target or with the grouping column — that is informative missingness, and it
# usually deserves an explicit indicator feature rather than plain imputation.

## 5. Univariate distributions

Shape, scale, skew, and suspicious spikes. Spikes at round numbers, at zero, or
at a sensor's range limit are usually encoded missing values in disguise.

In [ ]:
df.describe().T

# TODO: plot the distribution of each numeric feature and the value counts of
# each categorical feature. Note anything that looks like a sentinel value
# (-999, 0, exactly the range maximum).

## 6. Target analysis

The distribution of what you are predicting determines the metric, the loss,
and whether class imbalance or a long tail needs handling.

In [ ]:
target = cfg.target
df[target].describe()

# TODO: classification -> class balance; regression -> distribution and tail.
# TODO: decide the primary metric from what you see here, and record it in
#       src/evaluation/metrics.py as PRIMARY_METRIC.

## 7. Group structure

**This section decides how the data is split.**

Count how many records share each value of the candidate grouping column. If
records repeat per entity — the same device, trajectory, session, patient —
then a random split leaks correlated records across the boundary and every
downstream number is optimistic.

In [ ]:
group_col = cfg.split.group_col
sizes = df.groupby(group_col).size()
print(f"{sizes.size:,} groups, {sizes.mean():.1f} records per group on average")
sizes.describe()

# TODO: confirm group_col in configs/data.yaml is the right grouping key.
# TODO: check there are enough groups for the configured test_size to be stable
#       — a handful of large groups makes the split coarse and high-variance.

## 8. Relationships

Feature-target relationships worth modelling, and feature-feature correlations
worth knowing about. Watch for anything *too* predictive: a feature that
almost perfectly explains the target is usually leakage from the label.

In [ ]:
numeric = df.select_dtypes("number")
numeric.corr()[target].sort_values(ascending=False)

# TODO: inspect the top correlates — is any of them computed from the target,
#       recorded after it, or otherwise unavailable at prediction time?
# TODO: note strongly collinear pairs for the feature-selection step in 02x.

## 9. Temporal or spatial structure

Skip if the data has neither. If it has either, drift over time or across sites
changes what a fair split looks like and whether a domain-adaptation model is
worth trying.

In [ ]:
# TODO: plot the target and key features over time / across sites.
# TODO: if the distribution shifts, consider a time-based or site-based split
#       and record the decision in configs/data.yaml with a comment.

## 10. Data quality issues and candidate hard constraints

The output of this section feeds directly into notebook 01.

For every filter you propose, write down the **external source** of the bound.
If you cannot name one, it is a statistical threshold: it belongs in `02x`,
fitted on train only.

| Column | Rule | Source | Records affected |
|---|---|---|---|
| `<column>` | `<= <upper>` | `<standard / spec / physical limit>` | `<n>` |

In [ ]:
# TODO: quantify each candidate rule before adopting it — a rule that removes
#       30% of the data is a conversation, not a cleaning step.
# candidate = df["<column>"] > <upper-bound>
# print(f"{candidate.sum():,} records ({candidate.mean():.1%}) violate <rule>")

# Then declare each accepted rule in configs/data.yaml under schema.columns,
# including its `source` field, so notebook 01 can enforce it.

## 11. Findings — what 01 and 02x must do

Write this up properly. It is the specification the next notebooks implement,
and the reason anyone can trust the cleaning decisions six months from now.

**For notebook 01 (deterministic, model-agnostic):**
- Duplicates: `<definition of identity, expected count>`
- Columns to drop: `<...>`
- Invalid labels: `<rule>`
- Hard constraints: `<list, each with its source>`
- Split: `<method, grouping column, why>`

**For the `02x` notebooks (learned from train only):**
- Missing values: `<which columns, suggested strategy>`
- Statistical outliers: `<which columns, suggested method>`
- Scaling / encoding: `<...>`
- Feature engineering ideas: `<...>`

**Open questions:** `<anything that needs a domain expert>`